# Sample and null permute portion of every conversation.

Requested by @RickDale

In [1]:
import os
import numpy as np
import pandas as pd
import re
import shutil
from random import shuffle

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
SERVER_READY_DATA = '../data/server_ready'

OUTPUT_FOLDER = '../data/null-perm'
if not os.path.exists(OUTPUT_FOLDER):
    os.mkdir(OUTPUT_FOLDER)

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

## Iterate, permute

In [4]:
fs = grab_all_files(SERVER_READY_DATA)
len(fs)

26

In [5]:
sample_frac = .05

In [6]:
families = set([f.split('/')[-1].split('-')[1].replace('.parquet', '') for f in fs])
families

{'callfriend', 'callhome', 'croatian', 'finchat_corpus', 'yiddish'}

In [7]:
for fam in tqdm(families):

    familiars = [f for f in fs if (fam in f)]
    df = [
        pq.ParquetFile(f).read(columns=['corpus', 'conversation_id', 'x_turn_id', 'y_turn_id', 'x_speaker', 'y_speaker', 'x_utterance', 'y_utterance']).to_pandas()
        for f in familiars
    ]

    df = pd.concat(df, ignore_index=True)

    good_y = ~df['y_utterance'].isna()

    if 'yiddish' in df['corpus'].unique():
        df['corpus'] = 'yiddish-yid'

    df['lang'] = [corpus.split('-')[1] for corpus in df['corpus']]
    print(df['lang'].value_counts())

    df['output_file_name'] = 'xling-' + df['corpus'] + '-' + df['conversation_id'] + '.parquet'


    for lang in df['lang'].unique():
        df_ = df.loc[df['lang'].isin([lang]) & good_y]

        groups = df_.groupby(by=['output_file_name'])
        for out_f,dfi in groups:

            dfk = dfi.copy()
            dfk = dfk.groupby('x_turn_id').apply(lambda x: x.sample(n=int(np.ceil(sample_frac * len(x)))))

            shuffled_utts = np.random.choice(
                df_.index[
                    ~df_.index.isin(dfi.index.values)
                ].values,
                size=(len(dfk),),
                replace=(len(df_) - len(dfk)) < len(dfk)
            )
            # print(shuffled_utts)

            dfk['y_utterance'] = df_['y_utterance'].loc[shuffled_utts].values


            del dfk['output_file_name']
            dfk.index = range(len(dfk))

            # print(dfk[['x_turn_id', 'y_utterance']].head(n=2)),

            dfk.to_parquet(
                os.path.join(OUTPUT_FOLDER, out_f[0]),
                engine='fastparquet',
                compression='snappy'
            )

  0%|          | 0/5 [00:00<?, ?it/s]

lang
cro    11756419
Name: count, dtype: int64
lang
fin    100099
Name: count, dtype: int64
lang
spa    32093503
fra    11641191
ko      9170611
eng     6863556
zho     5244680
Name: count, dtype: int64
lang
eng    10444141
zho     6333029
deu     5473047
spa     3968514
Name: count, dtype: int64
lang
yid    21817
Name: count, dtype: int64


## Sample/Test

In [4]:
fs = grab_all_files(OUTPUT_FOLDER)
len(fs)

1290

In [5]:
for f in tqdm(np.random.choice(fs,size=(10,), replace=False)):
    df = pd.read_parquet(f)
    print(f.split('/')[-1])
    print(len(df))
    print(df.isna().sum())
    print(df['y_utterance'].sample(n=3).values.tolist())
    print('-----\n')

    # df.to_parquet(
    #     os.path.join('/Users/zacharyrosen/Desktop/', f.split('/')[-1]),
    #     engine='fastparquet',
    #     compression='snappy'
    # )
    # df.to_parquet(
    #     f,
    #     engine='fastparquet',
    #     compression='snappy'
    # )

  0%|          | 0/10 [00:00<?, ?it/s]

xling-callhome-zho-callhome-zho-1280.parquet
1794
corpus             0
conversation_id    0
x_turn_id          0
y_turn_id          0
x_speaker          0
y_speaker          0
x_utterance        0
y_utterance        0
lang               0
dtype: int64
['没有 了', '自己 吗 再 挣 这个 呐', '几 楼']
-----

xling-callfriend-zho-callfriend-zho-4336.parquet
7480
corpus             0
conversation_id    0
x_turn_id          0
y_turn_id          0
x_speaker          0
y_speaker          0
x_utterance        0
y_utterance        0
lang               0
dtype: int64
['是 这样 子 的 吗', '他 不 可能 惦 记 李 红 呢', '那 就是 什么']
-----

xling-callfriend-spa-c-callfriend-spa-c-6884.parquet
3256
corpus             0
conversation_id    0
x_turn_id          0
y_turn_id          0
x_speaker          0
y_speaker          0
x_utterance        0
y_utterance        0
lang               0
dtype: int64
['tú de dónde estás', 'breathes', 'ajá']
-----

xling-croation-cro-croation-cro-2011_77.parquet
3719
corpus             0
conversation_id  

## Sort and send

In [15]:
fs = [f for f in os.listdir(OUTPUT_FOLDER) if not f.startswith('.') and (not os.path.isdir(os.path.join(OUTPUT_FOLDER,f)))]
len(fs)

1290

In [16]:
to_rosen = np.random.choice(fs, size=(int(len(fs)/2),), replace=False).tolist()
to_itkin = [f for f in fs if f not in to_rosen]
len(to_itkin), len(to_rosen)

(645, 645)

In [17]:
ROSEN_PATH = os.path.join(OUTPUT_FOLDER, 'to_rosen')
if not os.path.exists(ROSEN_PATH):
    os.mkdir(ROSEN_PATH)

for f in tqdm(to_rosen):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ROSEN_PATH, f)
    )

  0%|          | 0/645 [00:00<?, ?it/s]

In [18]:
ITKIN_PATH = os.path.join(OUTPUT_FOLDER, 'to_itkin')
if not os.path.exists(ITKIN_PATH):
    os.mkdir(ITKIN_PATH)

for f in tqdm(to_itkin):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ITKIN_PATH, f)
    )

  0%|          | 0/645 [00:00<?, ?it/s]